# Setup

In [1]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# subsampling: percentage of commenting users to keep
USERS_PERCENTAGE = 0.1
assert 0 < USERS_PERCENTAGE <= 1, "invalid USER_PERCENTAGE, should be in (0, 1]"

# configuration
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_HASHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
assert LSH_NUM_HASHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_HASHES, must be equal to LSH_BANDS * LSH_ROWS"

In [2]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)

Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [3]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext

Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = ss.read.options(**csv_options).csv("dataset/nyt-articles-2020.csv")
full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-part0.csv") # TODO: this is a subset of all comments

full_articles.take(1), full_comments.take(1)

([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [5]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data)

Note: you may need to restart the kernel to use updated packages.


In [6]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))

articles.take(1), comments.take(1)

([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

## Utility Matrix

In [7]:
utility_matrix = comments.distinct() # TODO: keep multiple comments on same article or not?
utility_matrix.take(5), utility_matrix.count()

([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)],
 356283)

## Articles Profile - via dictionary

In [8]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ and len(x[0]) <= 100)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = sorted(pruned_keywords + (articles
            .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
            .filter(lambda x: len(x) < 100)
            .distinct()
            .collect()))

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting at most {((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = articles.flatMap(build_article_profile_dict)
articles_profile_dict.take(10), articles_profile_dict.count()

Extacted 3555 features (broadcasting at most 1402 KB)


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3395),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3465),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3193),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1131),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1097),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1158),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 628),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3194),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1086),
  ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', 3399)],
 158079)

## Articles Profile - via hashing

In [9]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = articles.flatMap(build_article_profile_hash)
articles_profile_hash.take(10), articles_profile_hash.count()

([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 653),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2704),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2692),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1483),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 4497),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 260),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1675),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2683),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3196),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 598)],
 185396)

### WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [10]:
articles_profile = articles_profile_hash

## Users Profile

In [11]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                     .join(articles_profile)
                     .map(lambda x: ((x[1][0], x[1][1]), 1))
                     .reduceByKey(lambda a, b: a + b))
users_profile.take(10), users_profile.count()

([((5646097, 535), 36),
  ((5646097, 2881), 36),
  ((5646097, 4221), 36),
  ((86273619, 535), 11),
  ((86273619, 2881), 11),
  ((86273619, 4221), 11),
  ((61536727, 535), 9),
  ((61536727, 2881), 9),
  ((61536727, 4221), 9),
  ((41508509, 535), 20)],
 2801463)

## Computing Similarity - LSH using hyperplanes (sketches)

In [12]:
import random

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_HASHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile
                  .map(lambda x: (x[0][0], ((x[0][1], x[1]))))
                  .groupByKey()
                  .mapValues(list)
                  .map(compute_sketch))
articles_sketches = (articles_profile
                  .map(lambda x: (x[0], ((x[1], 1))))
                  .groupByKey()
                  .mapValues(list)
                  .map(compute_sketch))

users_sketches.take(1), articles_sketches.take(1)

([(13120442,
   [0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
   

In [13]:
def lsh_buckets(row, what):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, (what, item_id)))
    return res

users_buckets = users_sketches.flatMap(lambda x: lsh_buckets(x, "USER"))
articles_buckets = articles_sketches.flatMap(lambda x: lsh_buckets(x, "ARTICLE"))

users_buckets.take(10), articles_buckets.take(10)

([(967667862, ('USER', 13120442)),
  (1223553998, ('USER', 13120442)),
  (1476691302, ('USER', 13120442)),
  (-697956923, ('USER', 13120442)),
  (-41903044, ('USER', 13120442)),
  (885152779, ('USER', 13120442)),
  (1028160308, ('USER', 13120442)),
  (-1521704519, ('USER', 13120442)),
  (-588454467, ('USER', 13120442)),
  (-2145614895, ('USER', 13120442))],
 [(1971969130,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (-407049733,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (-1947123645,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (-430573338,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (1999708629,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (643269541,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (-1900408999,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (-610594256,
   ('ARTICLE', 'nyt://

In [ ]:
# no distinct 20 bands x  5 rows = 1324640018 results in  3min 20s
# no distinct 10 bands x 10 rows =   58934055 results in       30s
#    distinct 10 bands x 10 rows =   32949487 results in  6min 50s

recommendations = (users_buckets
                   .join(articles_buckets)
                   .map(lambda x: (x[1][0][1], x[1][1][1])))
                   #.distinct())

recommendations.take(10), recommendations.count()

([(13120442, 'nyt://article/d5686346-4b09-56b4-89f3-fe1c9c291f13'),
  (13120442, 'nyt://article/ba3a7a40-33b9-599d-8e2e-db953a84707b'),
  (13120442, 'nyt://article/a5586c8e-f7f8-5d04-abd5-1967a573890f'),
  (13120442, 'nyt://article/6bc0b816-6248-5131-a4f2-cee9fae576f3'),
  (13120442, 'nyt://article/2e813ca4-4391-501a-b45a-2817ee3b79cc'),
  (13120442, 'nyt://article/03f8dfad-b2b5-5efa-a16e-0668c76d78df'),
  (13120442, 'nyt://article/3a20c986-3a92-5073-a2d9-8b9362f190f1'),
  (13120442, 'nyt://article/873c285b-9c53-539d-8766-c0d0a9aa69c0'),
  (13120442, 'nyt://article/52440aae-c975-5037-b38b-f41e8874ca73'),
  (13120442, 'nyt://article/0f606d66-8069-528e-97f9-ad52dc7d8fee')],
 64429495)

---
---

In [ ]:
# setup pandas (tests before scaling to spark)
%pip install pandas
import pandas as pd
articles = pd.read_csv("dataset/nyt-articles-2020.csv")
comments = pd.read_csv("dataset/nyt-comments-part0.csv") # TODO: this is already a sample
print(f"Dataset size: {len(articles)=}, {len(comments)=}")
(articles.columns, comments.columns)

In [ ]:
# subsampling
# WARNING: executing this cell multiple times without executing previous cell will continue to shrink the dataset
if USERS_PERCENTAGE < 1:
    total_users = len(comments['userID'].unique())
    subsampled_users = int(total_users * USERS_PERCENTAGE)
    top_users = comments['userID'].value_counts().nlargest(subsampled_users).index # keep top users
    print(top_users) # TODO

    print(f"Subsampling from {total_users} to {subsampled_users} users:")

    print(f"- comments: {len(comments)} -> ", end="")
    comments = comments[comments['userID'].isin(top_users)].copy() # keep only comments from these users
    print(f"{len(comments)}")

    active_article_ids = comments['articleID'].unique() # keep only commented articles
    print(f"- articles: {len(articles)} -> ", end="")
    articles = articles[articles['uniqueID'].isin(active_article_ids)].copy() # keep only commented articles
    print(f"{len(articles)}")

# Utility Matrix

In [ ]:
utility_matrix = pd.crosstab(index=comments['userID'], columns=comments['articleID'])
utility_matrix = (utility_matrix > 0).astype(int) # ignore multiple comments on the same article
utility_matrix

# Content-Based Recommendations

### Articles Profile

In [ ]:
# items profile
feature_columns = ["newsdesk", "section", "subsection"]
item_features_matrix = pd.get_dummies(articles[feature_columns], dtype=int)
item_features_matrix.index = articles['uniqueID'].values
# articles["profile"] = item_features_matrix.values.tolist()
item_features_matrix

### Users Profile

In [ ]:
# user profile: count how many times each user commented on articles with each label
user_profiles = []

for user_id in utility_matrix.index:
    commented_articles = utility_matrix.loc[user_id][utility_matrix.loc[user_id] == 1].index # articles commented by this user
    user_profile = item_features_matrix.loc[item_features_matrix.index.isin(commented_articles)].sum(axis=0) # sum features vectors for these articles
    user_profiles.append(user_profile)

user_features_matrix = pd.DataFrame(user_profiles, index=utility_matrix.index)
user_features_matrix

### Calculate Recommendations

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity # TODO: implement from scratch?
similarity_array = cosine_similarity(user_features_matrix, item_features_matrix)
similarity_df = pd.DataFrame(
    similarity_array,
    index=user_features_matrix.index,
    columns=item_features_matrix.index
)
similarity_df.to_csv("similarity_df.csv")
similarity_df

In [ ]:
articles_lookup = articles.set_index("uniqueID", drop=False)

def print_article(i, article_id, score=None):
    if article_id in articles_lookup.index:
        row = articles_lookup.loc[article_id]
        title = row.get("headline", row.get("title", "N/A"))
        abstract = row.get("abstract", "N/A")
        newsdesk = row.get("newsdesk", "N/A")
        section = row.get("section", "N/A")
        subsection = row.get("subsection", "N/A")
    else:
        title = abstract = newsdesk = section = subsection = "N/A"

    score_str = f" | score: {score:.5f}" if score is not None else ""
    print(f"{i}. article_id: {article_id}{score_str} | {title} | {abstract} | {newsdesk=} | {section=} | {subsection=}")

def print_results(user_id, recommendations=5):

    commented_ids = []
    if user_id in utility_matrix.index:
        commented = utility_matrix.loc[user_id]
        commented_ids = commented[commented > 0].index.tolist()

    scores = similarity_df.loc[user_id].sort_values(ascending=False)
    # remove alreayd commented articles
    scores = scores.drop(index=commented_ids, errors="ignore")


    print(f"=== Top {len(scores.head(recommendations))} recommendations for user {user_id}:")
    for i, (article_id, score) in enumerate(scores.head(recommendations).items(), start=1):
        print_article(i, article_id, score)

    print(f"\n=== Articles already commented by user {user_id} ({len(commented_ids)}):")
    for i, article_id in enumerate(commented_ids, start=1):
        print_article(i, article_id)

print_results(similarity_df.index[12], 50)
